In [1]:
import numpy as np
import torch

from pfi import hyperopt_pfi
from pfi.utils.data import load_data, X_from_snapshots

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device


from examples.mpl_config import apply_mpl_config
apply_mpl_config()



/mnt/home/vchardes/.local/share/venvs/score/lib/python3.10/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


device(type='cuda')

In [2]:
seed = 0
np.random.seed(seed)
torch.manual_seed(seed)

nsamples = 3000

path = '/mnt/home/vchardes/ceph/datasets/HSC_data/10XChromiumV3_10.1038_s41467-021-27159-x_10.5281_zenodo.5291737_exvivo.h5ad'
genes = ['gata1','fli1','klf1','gata2','gfi1','gfi1b','runx1','tal1','jun','spi1','zfpm1','lmo2','etv6','erg','cebpa',
         'meis1','sall4','myc','foxo3','zbtb7a','nanog','nfe2','stat3','mef2c']

samples, times, ind_array, cell_types = load_data(
    path=path,
    nsamples=nsamples,
    genes=genes,
    time_key='day',
    cell_type_key='cell_type',
    seed=seed,
)

X = X_from_snapshots([samples[k] for k in range(samples.shape[0])], times)
ndim = X.shape[1] - 1
print('X shape:', X.shape, 'ndim:', ndim, 'times:', times)


X shape: (18000, 25) ndim: 24 times: [ 0  2  4  6  8 11]


In [ ]:
# Optional: provide Optuna distributions in search_space
# from optuna.distributions import CategoricalDistribution, IntDistribution, FloatDistribution
# search_space = {
#     "s_width": CategoricalDistribution([64, 128]),
#     "f_width": CategoricalDistribution([64, 128, 256]),
#     "s_depth": IntDistribution(2, 4, step=1),
#     "s_noise_lvl": FloatDistribution(0.05, 0.4),
# }
search_space = None

# Run Optuna hyperparameter optimization (default search space when None)
study = hyperopt_pfi(
    X,
    n_trials=10,
    search_space=search_space,
    device=device,
    seed=seed,
)

print('Best trials (Pareto front):')
for t in study.best_trials:
    print('trial', t.number, 'values', t.values, 'params', t.params)


[I 2026-03-06 10:47:09,981] A new study created in memory with name: no-name-d3fe80ff-816a-48e1-a2c8-7d0d38b4d1c2


  0%|          | 0/10 [00:00<?, ?it/s]

[I 2026-03-06 10:49:28,144] Trial 0 finished with values: [0.13332358996073404, 13.470259666442871] and parameters: {'solver': 'pfm', 's_net': 'dnn', 'f_net': 'dnn', 's_width': 64, 'f_width': 64, 's_depth': 3, 'f_depth': 4, 'noise_lvl': 0.4842064267814707, 'lx': 1.2000000000000002, 'L': 5, 'fac': 2, 'nb': 1, 's_lr': 0.001, 'f_lr': 0.001, 's_n_epochs': 5000, 'f_n_epochs': 4000, 'fit_on_score_samples': True}.
[I 2026-03-06 10:51:30,300] Trial 1 finished with values: [0.13342666625976562, 0.26616411209106444] and parameters: {'solver': 'pfm', 's_net': 'dnn', 'f_net': 'spectral', 's_width': 64, 'f_width': 128, 's_depth': 3, 'f_depth': 2, 'noise_lvl': 0.4070763430038196, 'lx': 0.8, 'L': 5, 'fac': 2, 'nb': 1, 's_lr': 0.001, 'f_lr': 0.001, 's_n_epochs': 5000, 'f_n_epochs': 4000, 'fit_on_score_samples': True}.
[I 2026-03-06 10:54:18,522] Trial 2 finished with values: [0.14926910400390625, 0.2407533645629883] and parameters: {'solver': 'pfm', 's_net': 'dnn', 'f_net': 'spectral', 's_width': 64, 

In [ ]:
from plotly.io import show
import optuna
fig = optuna.visualization.plot_pareto_front(study)
show(fig)